In [1]:
# ==========================================
# COMPLETE PyQt6 + RASTERIO TOOL
# ALL FEATURES FROM ORIGINAL TKINTER VERSION
# Smart Band Detection | All Indices Support
# NDSI | NDWI | NDVI | Batch Processing
# Model Prediction Tab
# ==========================================

from PyQt6.QtWidgets import *
from PyQt6.QtCore import *
from PyQt6.QtGui import *
import rasterio
import numpy as np
import os
import sys
import platform
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from reportlab.platypus import SimpleDocTemplate, Paragraph, Image
from reportlab.lib.styles import getSampleStyleSheet
from PIL import Image as PILImage
import io

# Platform detection
PLATFORM_SYSTEM = platform.system()
IS_WINDOWS = PLATFORM_SYSTEM == 'Windows'
IS_MAC = PLATFORM_SYSTEM == 'Darwin'
IS_LINUX = PLATFORM_SYSTEM == 'Linux'

# ----------------GLOBALS ----------------
image_path = None
result_data = None
profile = None
green = None
swir = None
ndsi_threshold = 0.4
folder_path = None
image_files = []
current_image_index = 0
output_folder_path = None
loaded_model = None
model_path = None
_band_cache = {}
model_input_image_path = None
model_prediction_result = None
model_batch_folder = None
model_output_folder = None

# ---------------- COLOR PALETTE ----------------
COLORS = {
    'background': '#0f1117',
    'card_bg': '#1a1d27',
    'card_border': '#2a2e3a',
    'primary': '#5b8cfa',
    'primary_hover': '#7aa3fc',
    'primary_disabled': '#3a4155',
    'secondary_bg': '#2e3347',
    'secondary_hover': '#3b4260',
    'secondary_border': '#5a6478',
    'text': '#e8eaf0',
    'text_muted': '#6b7280',
    'text_link': '#5b8cfa',
    'success': '#4ade80',
    'warning': '#fbbf24',
    'error': '#f87171',
    'canvas_bg': '#111827',
    'listbox_bg': '#141620',
    'listbox_sel': '#5b8cfa',
    'details_bg': '#141620',
}

_FONT_TITLE_SIZE = 26
_FONT_SUB_SIZE = 11
_FONT_SECTION_SIZE = 12
_FONT_BODY_SIZE = 10
_FONT_SMALL_SIZE = 9
_FONT_BTN_SIZE = 11
_FONT_BADGE_SIZE = 10
_FONT_FAMILY = "SF Pro Display" if IS_MAC else "Segoe UI"


# ============================================================
# BAND DETECTION FUNCTIONS
# ============================================================
def detect_band_by_type(src, band_type):
    if src is None or src.closed:
        return None
    try:
        band_descriptions = src.descriptions
        band_count = src.count
    except Exception:
        return None

    patterns = {
        'GREEN': ['B3', 'GREEN', 'GRN', 'BAND3', 'BAND 3', 'SR_B3'],
        'RED': ['B4', 'RED', 'BAND4', 'BAND 4', 'SR_B4'],
        'NIR': ['B8', 'NIR', 'BAND8', 'BAND 8', 'B8A', 'SR_B8', 'NEAR_INFRARED'],
        'SWIR': ['B11', 'SWIR', 'SWIR1', 'BAND11', 'BAND 11', 'SR_B11'],
        'BLUE': ['B2', 'BLUE', 'BLU', 'BAND2', 'BAND 2', 'SR_B2']
    }

    if band_type not in patterns:
        return None

    if band_descriptions:
        for i, desc in enumerate(band_descriptions):
            if desc:
                desc_upper = str(desc).upper().strip()
                for pattern in patterns[band_type]:
                    if desc_upper == pattern.upper():
                        return i + 1
                for pattern in patterns[band_type]:
                    if pattern.upper() in desc_upper:
                        return i + 1
    return None


def _cached_detect_band(src, band_type):
    global _band_cache
    if src is None or src.closed:
        return None
    try:
        key = src.name
        if key not in _band_cache:
            _band_cache[key] = {}
        cache = _band_cache[key]
        if band_type not in cache:
            cache[band_type] = detect_band_by_type(src, band_type)
        return cache[band_type]
    except Exception:
        return detect_band_by_type(src, band_type)


def get_required_bands(src, index_type):
    result = {'success': False, 'bands': {}, 'info': '', 'missing': []}

    if index_type == 'NDSI':
        required = ['GREEN', 'SWIR']
    elif index_type == 'NDWI':
        required = ['GREEN', 'NIR']
    elif index_type == 'NDVI':
        required = ['NIR', 'RED']
    else:
        result['info'] = f"Unknown index type: {index_type}"
        return result

    band_info_list = []
    for band_type in required:
        band_idx = _cached_detect_band(src, band_type)
        if band_idx:
            band_data = src.read(band_idx).astype(float)
            result['bands'][band_type] = {
                'data': band_data,
                'index': band_idx,
                'name': src.descriptions[band_idx - 1] if src.descriptions and src.descriptions[
                    band_idx - 1] else f"Band {band_idx}"
            }
            band_info_list.append(f"{band_type}: Band {band_idx} ({result['bands'][band_type]['name']})")
        else:
            result['missing'].append(band_type)

    if len(result['bands']) == len(required):
        result['success'] = True
        result['info'] = "✓ All bands detected:\n" + "\n".join(band_info_list)
    else:
        result['info'] = f"✗ Missing bands: {', '.join(result['missing'])}\n"
        if band_info_list:
            result['info'] += "Found:\n" + "\n".join(band_info_list)

    return result


# ============================================================
# CALCULATION FUNCTIONS
# ============================================================
def calculate_ndsi(green, swir):
    numerator = green - swir
    denominator = green + swir
    with np.errstate(divide='ignore', invalid='ignore'):
        ndsi = np.where(denominator != 0, numerator / denominator, 0)
    return ndsi


def calculate_ndwi(green, nir):
    numerator = green - nir
    denominator = green + nir
    with np.errstate(divide='ignore', invalid='ignore'):
        ndwi = np.where(denominator != 0, numerator / denominator, 0)
    return ndwi


def calculate_ndvi(nir, red):
    numerator = nir - red
    denominator = nir + red
    with np.errstate(divide='ignore', invalid='ignore'):
        ndvi = np.where(denominator != 0, numerator / denominator, 0)
    return ndvi


# ============================================================
# CUSTOM KERAS METRICS
# ============================================================
def dice_coef(y_true, y_pred, smooth=1):
    """
    Dice coefficient for semantic segmentation
    Compatible with all TensorFlow versions
    """
    try:
        import tensorflow as tf
        y_true_f = tf.keras.backend.flatten(y_true)
        y_pred_f = tf.keras.backend.flatten(y_pred)
        intersection = tf.keras.backend.sum(y_true_f * y_pred_f)
        return (2. * intersection + smooth) / (tf.keras.backend.sum(y_true_f) + tf.keras.backend.sum(y_pred_f) + smooth)
    except:
        pass

def dice_loss(y_true, y_pred, smooth=1):
    """Dice loss function"""
    return 1 - dice_coef(y_true, y_pred, smooth)

def iou_coef(y_true, y_pred, smooth=1):
    """IoU coefficient"""
    try:
        import tensorflow as tf
        y_true_f = tf.keras.backend.flatten(y_true)
        y_pred_f = tf.keras.backend.flatten(y_pred)
        intersection = tf.keras.backend.sum(y_true_f * y_pred_f)
        union = tf.keras.backend.sum(y_true_f) + tf.keras.backend.sum(y_pred_f) - intersection
        return (intersection + smooth) / (union + smooth)
    except:
        pass


# ============================================================
# MAIN WINDOW CLASS
# ============================================================
class MainWindow(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Raster Index Calculator – PyQt6 Professional Edition")
        self.setMinimumSize(1600, 1000)

        # Widget references
        self.status_label = None
        self.btn_run = None
        self.threshold_label = None
        self.threshold_value_label = None
        self.threshold_slider = None
        self.preview_label = None
        self.image_list = None
        self.image_info_label = None
        self.calc_combo = None
        self.folder_label = None
        self.folder_info_label = None
        self.file_name_label = None
        self.file_info_label = None
        self.output_folder_display = None
        self.model_filename_display = None
        self.model_details_text = None
        self.model_preview_label = None
        self.model_result_label = None

        # Setup UI
        self.setup_dark_theme()
        self.init_ui()

    def setup_dark_theme(self):
        palette = QPalette()
        palette.setColor(QPalette.ColorRole.Window, QColor(COLORS['background']))
        palette.setColor(QPalette.ColorRole.WindowText, QColor(COLORS['text']))
        palette.setColor(QPalette.ColorRole.Base, QColor(COLORS['card_bg']))
        palette.setColor(QPalette.ColorRole.AlternateBase, QColor(COLORS['listbox_bg']))
        palette.setColor(QPalette.ColorRole.Text, QColor(COLORS['text']))
        palette.setColor(QPalette.ColorRole.Button, QColor(COLORS['secondary_bg']))
        palette.setColor(QPalette.ColorRole.ButtonText, QColor(COLORS['text']))
        palette.setColor(QPalette.ColorRole.Highlight, QColor(COLORS['primary']))
        palette.setColor(QPalette.ColorRole.HighlightedText, QColor('#ffffff'))
        self.setPalette(palette)

        self.setStyleSheet(f"""
            QMainWindow {{
                background-color: {COLORS['background']};
            }}
            QWidget {{
                background-color: {COLORS['background']};
                color: {COLORS['text']};
            }}
            QPushButton {{
                background-color: {COLORS['secondary_bg']};
                color: #ffffff;
                border: none;
                border-radius: 8px;
                padding: 10px;
                font-size: {_FONT_BTN_SIZE}pt;
                font-weight: bold;
            }}
            QPushButton:hover {{
                background-color: {COLORS['secondary_hover']};
            }}
            QPushButton:disabled {{
                background-color: {COLORS['primary_disabled']};
                color: #6b7280;
            }}
            QPushButton.primary {{
                background-color: {COLORS['primary']};
            }}
            QPushButton.primary:hover {{
                background-color: {COLORS['primary_hover']};
            }}
            QLabel {{
                color: {COLORS['text']};
                background-color: transparent;
            }}
            QFrame {{
                background-color: {COLORS['card_bg']};
                border: 1px solid {COLORS['card_border']};
                border-radius: 12px;
            }}
            QListWidget {{
                background-color: {COLORS['listbox_bg']};
                color: {COLORS['text']};
                border: 1px solid {COLORS['card_border']};
                border-radius: 8px;
                padding: 8px;
            }}
            QListWidget::item:selected {{
                background-color: {COLORS['primary']};
            }}
            QTextEdit {{
                background-color: {COLORS['details_bg']};
                color: {COLORS['text']};
                border: 1px solid {COLORS['card_border']};
                border-radius: 8px;
                padding: 12px;
            }}
            QComboBox {{
                background-color: {COLORS['listbox_bg']};
                color: {COLORS['text']};
                border: 1px solid {COLORS['card_border']};
                border-radius: 8px;
                padding: 8px;
            }}
            QComboBox:hover {{
                background-color: {COLORS['secondary_hover']};
            }}
            QComboBox::drop-down {{
                border: none;
            }}
            QComboBox QAbstractItemView {{
                background-color: {COLORS['secondary_bg']};
                color: {COLORS['text']};
                selection-background-color: {COLORS['primary']};
            }}
            QSlider::groove:horizontal {{
                background: {COLORS['card_border']};
                height: 6px;
                border-radius: 3px;
            }}
            QSlider::handle:horizontal {{
                background: {COLORS['primary']};
                width: 18px;
                margin: -6px 0;
                border-radius: 9px;
            }}
            QSlider::handle:horizontal:hover {{
                background: {COLORS['primary_hover']};
            }}
            QTabWidget::pane {{
                border: 1px solid {COLORS['card_border']};
                background-color: {COLORS['background']};
            }}
            QTabBar::tab {{
                background-color: {COLORS['card_bg']};
                color: {COLORS['text_muted']};
                padding: 12px 24px;
                border: 1px solid {COLORS['card_border']};
                border-bottom: none;
                border-top-left-radius: 8px;
                border-top-right-radius: 8px;
                margin-right: 4px;
            }}
            QTabBar::tab:selected {{
                background-color: {COLORS['secondary_bg']};
                color: {COLORS['text']};
                font-weight: bold;
            }}
            QScrollArea {{
                border: none;
                background-color: transparent;
            }}
            QScrollBar:vertical {{
                background-color: {COLORS['background']};
                width: 12px;
                border-radius: 6px;
            }}
            QScrollBar::handle:vertical {{
                background-color: {COLORS['card_border']};
                border-radius: 6px;
            }}
            QScrollBar::handle:vertical:hover {{
                background-color: {COLORS['secondary_bg']};
            }}
        """)

    def init_ui(self):
        # Central widget
        central_widget = QWidget()
        self.setCentralWidget(central_widget)
        main_layout = QVBoxLayout(central_widget)
        main_layout.setContentsMargins(28, 18, 28, 18)
        main_layout.setSpacing(14)

        # Header
        header_frame = QFrame()
        header_frame.setStyleSheet(f"""
            QFrame {{
                background-color: {COLORS['card_bg']};
                border: none;
                border-bottom: 1px solid {COLORS['card_border']};
                border-radius: 0px;
            }}
        """)
        header_layout = QVBoxLayout(header_frame)
        header_layout.setContentsMargins(0, 14, 0, 14)

        title = QLabel("🛰️  Raster Index Calculator")
        title.setFont(QFont(_FONT_FAMILY, _FONT_TITLE_SIZE, QFont.Weight.Bold))
        title.setAlignment(Qt.AlignmentFlag.AlignCenter)
        header_layout.addWidget(title)

        subtitle = QLabel("Professional GeoTIFF Analysis Tool with Smart Band Detection")
        subtitle.setFont(QFont(_FONT_FAMILY, _FONT_SUB_SIZE))
        subtitle.setAlignment(Qt.AlignmentFlag.AlignCenter)
        subtitle.setStyleSheet(f"color: {COLORS['text_muted']};")
        header_layout.addWidget(subtitle)

        main_layout.addWidget(header_frame)

        # Content area - two columns
        content_widget = QWidget()
        content_layout = QGridLayout(content_widget)
        content_layout.setSpacing(12)

        # LEFT COLUMN (scrollable)
        left_scroll = QScrollArea()
        left_scroll.setWidgetResizable(True)
        left_scroll.setFrameShape(QFrame.Shape.NoFrame)
        left_widget = QWidget()
        left_layout = QVBoxLayout(left_widget)
        left_layout.setSpacing(12)

        # Card 1: Folders
        folders_card = self.create_card("Folders", "📁")
        folders_layout = QVBoxLayout()

        upload_info = QLabel("Select input and output folders")
        upload_info.setFont(QFont(_FONT_FAMILY, _FONT_SMALL_SIZE))
        upload_info.setStyleSheet(f"color: {COLORS['text_muted']};")
        folders_layout.addWidget(upload_info)

        input_label = QLabel("Input Folder:")
        input_label.setFont(QFont(_FONT_FAMILY, 9, QFont.Weight.Bold))
        folders_layout.addWidget(input_label)

        self.folder_label = QLabel("No folder selected")
        self.folder_label.setFont(QFont(_FONT_FAMILY, _FONT_BODY_SIZE))
        self.folder_label.setStyleSheet(f"color: {COLORS['text_muted']};")
        folders_layout.addWidget(self.folder_label)

        self.folder_info_label = QLabel("")
        self.folder_info_label.setFont(QFont(_FONT_FAMILY, _FONT_SMALL_SIZE))
        self.folder_info_label.setStyleSheet(f"color: {COLORS['text_muted']};")
        folders_layout.addWidget(self.folder_info_label)

        # Folder buttons
        folder_btn_layout = QHBoxLayout()
        btn_input = QPushButton("📂  Select Input Folder")
        btn_input.clicked.connect(self.upload_folder)
        folder_btn_layout.addWidget(btn_input)

        btn_output = QPushButton("💾  Select Output Folder")
        btn_output.clicked.connect(self.select_output_folder)
        folder_btn_layout.addWidget(btn_output)

        folders_layout.addLayout(folder_btn_layout)

        # Output folder display
        output_label = QLabel("Output Folder:")
        output_label.setFont(QFont(_FONT_FAMILY, 9, QFont.Weight.Bold))
        folders_layout.addWidget(output_label)

        self.output_folder_display = QLabel("Not selected")
        self.output_folder_display.setFont(QFont(_FONT_FAMILY, _FONT_BODY_SIZE))
        self.output_folder_display.setStyleSheet(f"color: {COLORS['text_muted']};")
        folders_layout.addWidget(self.output_folder_display)

        # Image listbox
        list_label = QLabel("Images in folder:")
        list_label.setFont(QFont(_FONT_FAMILY, 9, QFont.Weight.Bold))
        folders_layout.addWidget(list_label)

        self.image_list = QListWidget()
        self.image_list.setMinimumHeight(150)
        self.image_list.itemClicked.connect(self.load_selected_image)
        folders_layout.addWidget(self.image_list)

        # Clear selection button
        btn_clear = QPushButton("🔄 Clear Selection (Batch Mode)")
        btn_clear.clicked.connect(self.clear_image_selection)
        folders_layout.addWidget(btn_clear)

        folders_card.setLayout(folders_layout)
        left_layout.addWidget(folders_card)

        # Card 2: Current Image
        image_card = self.create_card("Current Image", "📄")
        image_layout = QVBoxLayout()

        self.file_name_label = QLabel("No image selected")
        self.file_name_label.setFont(QFont(_FONT_FAMILY, _FONT_BODY_SIZE))
        self.file_name_label.setStyleSheet(f"color: {COLORS['text_muted']};")
        self.file_name_label.setWordWrap(True)
        image_layout.addWidget(self.file_name_label)

        self.file_info_label = QLabel("")
        self.file_info_label.setFont(QFont(_FONT_FAMILY, _FONT_SMALL_SIZE))
        self.file_info_label.setStyleSheet(f"color: {COLORS['text_muted']};")
        image_layout.addWidget(self.file_info_label)

        image_card.setLayout(image_layout)
        left_layout.addWidget(image_card)

        # Card 3: Calculation
        calc_card = self.create_card("Calculation", "⚙️")
        calc_layout = QVBoxLayout()

        calc_type_label = QLabel("Index Type:")
        calc_type_label.setFont(QFont(_FONT_FAMILY, 10, QFont.Weight.Bold))
        calc_layout.addWidget(calc_type_label)

        self.calc_combo = QComboBox()
        self.calc_combo.addItems(["NDSI", "NDWI", "NDVI", "All Indices"])
        self.calc_combo.setFont(QFont(_FONT_FAMILY, _FONT_BODY_SIZE))
        self.calc_combo.currentTextChanged.connect(self.on_calculation_change)
        calc_layout.addWidget(self.calc_combo)

        # Threshold section
        threshold_header_layout = QHBoxLayout()
        self.threshold_label = QLabel("NDSI Threshold:")
        self.threshold_label.setFont(QFont(_FONT_FAMILY, 10, QFont.Weight.Bold))
        threshold_header_layout.addWidget(self.threshold_label)

        self.threshold_value_label = QLabel("0.40")
        self.threshold_value_label.setFont(QFont(_FONT_FAMILY, _FONT_BADGE_SIZE, QFont.Weight.Bold))
        self.threshold_value_label.setStyleSheet(f"""
            background-color: {COLORS['primary']};
            color: #ffffff;
            padding: 4px 10px;
            border-radius: 6px;
        """)
        self.threshold_value_label.setAlignment(Qt.AlignmentFlag.AlignCenter)
        self.threshold_value_label.setFixedWidth(60)
        threshold_header_layout.addWidget(self.threshold_value_label)

        calc_layout.addLayout(threshold_header_layout)

        self.threshold_slider = QSlider(Qt.Orientation.Horizontal)
        self.threshold_slider.setMinimum(0)
        self.threshold_slider.setMaximum(100)
        self.threshold_slider.setValue(40)
        self.threshold_slider.valueChanged.connect(self.on_threshold_change)
        calc_layout.addWidget(self.threshold_slider)

        calc_card.setLayout(calc_layout)
        left_layout.addWidget(calc_card)

        # Card 4: Actions
        action_card = self.create_card("Actions", "🚀")
        action_layout = QVBoxLayout()

        self.btn_run = QPushButton("▶  Run Calculation")
        self.btn_run.setProperty("class", "primary")
        self.btn_run.setMinimumHeight(50)
        self.btn_run.clicked.connect(self.run_calculation)
        self.btn_run.setEnabled(False)
        action_layout.addWidget(self.btn_run)

        action_card.setLayout(action_layout)
        left_layout.addWidget(action_card)

        left_layout.addStretch()
        left_scroll.setWidget(left_widget)
        content_layout.addWidget(left_scroll, 0, 0, 1, 1)

        # RIGHT COLUMN - Tabbed Interface
        right_widget = QWidget()
        right_layout = QVBoxLayout(right_widget)
        right_layout.setContentsMargins(0, 0, 0, 0)

        # Create Tab Widget
        self.tabs = QTabWidget()
        right_layout.addWidget(self.tabs)

        # Tab 1: Image Preview
        preview_tab = QWidget()
        preview_tab_layout = QVBoxLayout(preview_tab)
        preview_tab_layout.setContentsMargins(6, 6, 6, 6)

        preview_card = self.create_card("Preview", "🖼️")
        preview_layout = QVBoxLayout()

        self.preview_label = QLabel("No image loaded\n\n📂 Select an image to see preview")
        self.preview_label.setAlignment(Qt.AlignmentFlag.AlignCenter)
        self.preview_label.setMinimumHeight(600)
        self.preview_label.setStyleSheet(f"""
            background-color: {COLORS['canvas_bg']};
            color: {COLORS['text_muted']};
            border: 1px solid {COLORS['card_border']};
            border-radius: 8px;
            font-size: 12pt;
        """)
        preview_layout.addWidget(self.preview_label)

        self.image_info_label = QLabel("")
        self.image_info_label.setFont(QFont(_FONT_FAMILY, _FONT_SMALL_SIZE))
        self.image_info_label.setStyleSheet(f"color: {COLORS['text_muted']};")
        self.image_info_label.setWordWrap(True)
        preview_layout.addWidget(self.image_info_label)

        preview_card.setLayout(preview_layout)
        preview_tab_layout.addWidget(preview_card)
        
        self.tabs.addTab(preview_tab, "🖼️  Image Preview")

        # Tab 2: Model Tab
        self.create_model_tab()

        content_layout.addWidget(right_widget, 0, 1, 1, 1)

        # Set column stretches
        content_layout.setColumnStretch(0, 2)
        content_layout.setColumnStretch(1, 3)

        main_layout.addWidget(content_widget)

        # Status bar
        status_frame = QFrame()
        status_frame.setStyleSheet(f"""
            QFrame {{
                background-color: {COLORS['card_bg']};
                border: 1px solid {COLORS['card_border']};
                border-radius: 8px;
            }}
        """)
        status_layout = QHBoxLayout(status_frame)
        status_layout.setContentsMargins(20, 12, 20, 12)

        self.status_label = QLabel("⚪  Ready – Select input folder to begin")
        self.status_label.setFont(QFont(_FONT_FAMILY, _FONT_SUB_SIZE, QFont.Weight.Bold))
        self.status_label.setStyleSheet(f"color: {COLORS['text_muted']};")
        status_layout.addWidget(self.status_label)
        
        # Add fixed spacing between label and progress bar
        status_layout.addSpacing(50)
        
        # Progress bar - red style
        self.progress_bar = QProgressBar()
        self.progress_bar.setMinimum(0)
        self.progress_bar.setMaximum(100)
        self.progress_bar.setValue(0)
        self.progress_bar.setTextVisible(False)
        self.progress_bar.setFixedWidth(450)
        self.progress_bar.setFixedHeight(28)
        self.progress_bar.setStyleSheet("""
            QProgressBar {
                border: 2px solid #e0e0e0;
                border-radius: 14px;
                background-color: #f0f0f0;
            }
            QProgressBar::chunk {
                background-color: #ff4444;
                border-radius: 12px;
                margin: 1px;
            }
        """)
        self.progress_bar.hide()
        status_layout.addWidget(self.progress_bar)
        
        # Add stretch after progress bar to push it left
        status_layout.addStretch(1)

        main_layout.addWidget(status_frame)

        # Footer
        footer = QLabel("Powered by Rasterio & PyQt6  |  Smart Band Detection  ©  2026")
        footer.setFont(QFont(_FONT_FAMILY, _FONT_SMALL_SIZE))
        footer.setAlignment(Qt.AlignmentFlag.AlignCenter)
        footer.setStyleSheet(f"color: {COLORS['text_muted']}; padding: 10px;")
        main_layout.addWidget(footer)

    def create_card(self, title, icon):
        card = QFrame()
        card.setStyleSheet(f"""
            QFrame {{
                background-color: {COLORS['card_bg']};
                border: 1px solid {COLORS['card_border']};
                border-radius: 12px;
                padding: 16px;
            }}
        """)
        return card

    def create_model_tab(self):
        """Create the Model/ML tab with all sections"""
        model_tab = QWidget()
        model_tab_layout = QVBoxLayout(model_tab)
        model_tab_layout.setContentsMargins(6, 6, 6, 6)

        # Scrollable area
        scroll = QScrollArea()
        scroll.setWidgetResizable(True)
        scroll.setFrameShape(QFrame.Shape.NoFrame)
        scroll_widget = QWidget()
        scroll_layout = QVBoxLayout(scroll_widget)
        scroll_layout.setSpacing(12)

        # === SECTION 1: Model Upload ===
        model_upload_card = self.create_card("Machine Learning Model", "🤖")
        model_upload_layout = QVBoxLayout()

        desc_label = QLabel("Upload your trained Keras model (.keras, .h5) for classification")
        desc_label.setFont(QFont(_FONT_FAMILY, _FONT_BODY_SIZE))
        desc_label.setStyleSheet(f"color: {COLORS['text_muted']};")
        desc_label.setWordWrap(True)
        model_upload_layout.addWidget(desc_label)

        selected_label = QLabel("Selected Model:")
        selected_label.setFont(QFont(_FONT_FAMILY, 10, QFont.Weight.Bold))
        model_upload_layout.addWidget(selected_label)

        self.model_filename_display = QLabel("No model loaded")
        self.model_filename_display.setFont(QFont(_FONT_FAMILY, _FONT_BODY_SIZE))
        self.model_filename_display.setStyleSheet(f"""
            background-color: {COLORS['listbox_bg']};
            color: {COLORS['text_muted']};
            padding: 12px;
            border: 1px solid {COLORS['card_border']};
            border-radius: 8px;
        """)
        model_upload_layout.addWidget(self.model_filename_display)

        btn_upload_model = QPushButton("📤  Upload Keras Model  (.keras / .h5)")
        btn_upload_model.setProperty("class", "primary")
        btn_upload_model.setMinimumHeight(45)
        btn_upload_model.clicked.connect(self.upload_model)
        model_upload_layout.addWidget(btn_upload_model)

        details_label = QLabel("Model Details:")
        details_label.setFont(QFont(_FONT_FAMILY, 10, QFont.Weight.Bold))
        model_upload_layout.addWidget(details_label)

        self.model_details_text = QTextEdit()
        self.model_details_text.setMinimumHeight(150)
        self.model_details_text.setReadOnly(True)
        self.model_details_text.setPlainText("No model loaded yet.\n\nUpload a .keras or .h5 file to see model architecture and details.")
        model_upload_layout.addWidget(self.model_details_text)

        model_upload_card.setLayout(model_upload_layout)
        scroll_layout.addWidget(model_upload_card)

        # === SECTION 2: Single Image Prediction ===
        predict_card = self.create_card("Single Image Prediction", "🎯")
        predict_layout = QVBoxLayout()

        desc2 = QLabel("Select an image and run prediction")
        desc2.setFont(QFont(_FONT_FAMILY, _FONT_BODY_SIZE))
        desc2.setStyleSheet(f"color: {COLORS['text_muted']};")
        predict_layout.addWidget(desc2)

        btn_select_image = QPushButton("📷  Select Image for Prediction")
        btn_select_image.clicked.connect(self.select_model_input_image)
        predict_layout.addWidget(btn_select_image)

        self.model_preview_label = QLabel("No image selected")
        self.model_preview_label.setAlignment(Qt.AlignmentFlag.AlignCenter)
        self.model_preview_label.setMinimumHeight(200)
        self.model_preview_label.setStyleSheet(f"""
            background-color: {COLORS['canvas_bg']};
            color: {COLORS['text_muted']};
            border: 1px solid {COLORS['card_border']};
            border-radius: 8px;
        """)
        predict_layout.addWidget(self.model_preview_label)

        btn_predict = QPushButton("▶  Run Prediction")
        btn_predict.setProperty("class", "primary")
        btn_predict.setMinimumHeight(45)
        btn_predict.clicked.connect(self.run_model_prediction)
        predict_layout.addWidget(btn_predict)

        result_label = QLabel("Prediction Result:")
        result_label.setFont(QFont(_FONT_FAMILY, 10, QFont.Weight.Bold))
        predict_layout.addWidget(result_label)

        self.model_result_label = QLabel("No prediction yet")
        self.model_result_label.setFont(QFont(_FONT_FAMILY, _FONT_BODY_SIZE))
        self.model_result_label.setStyleSheet(f"""
            background-color: {COLORS['listbox_bg']};
            color: {COLORS['text_muted']};
            padding: 12px;
            border: 1px solid {COLORS['card_border']};
            border-radius: 8px;
        """)
        self.model_result_label.setWordWrap(True)
        predict_layout.addWidget(self.model_result_label)

        btn_save_pred = QPushButton("💾  Save Prediction Result")
        btn_save_pred.clicked.connect(self.save_prediction_result)
        predict_layout.addWidget(btn_save_pred)

        predict_card.setLayout(predict_layout)
        scroll_layout.addWidget(predict_card)

        # === SECTION 3: Batch Prediction ===
        batch_pred_card = self.create_card("Batch Prediction", "📦")
        batch_pred_layout = QVBoxLayout()

        desc3 = QLabel("Run predictions on multiple images")
        desc3.setFont(QFont(_FONT_FAMILY, _FONT_BODY_SIZE))
        desc3.setStyleSheet(f"color: {COLORS['text_muted']};")
        batch_pred_layout.addWidget(desc3)

        btn_batch_folder = QPushButton("📁  Select Batch Folder")
        btn_batch_folder.clicked.connect(self.select_batch_folder)
        batch_pred_layout.addWidget(btn_batch_folder)

        btn_model_output = QPushButton("💾  Select Output Folder")
        btn_model_output.clicked.connect(self.select_model_output_folder)
        batch_pred_layout.addWidget(btn_model_output)

        btn_batch_predict = QPushButton("🚀  Run Batch Prediction")
        btn_batch_predict.setProperty("class", "primary")
        btn_batch_predict.setMinimumHeight(45)
        btn_batch_predict.clicked.connect(self.run_batch_prediction)
        batch_pred_layout.addWidget(btn_batch_predict)

        batch_pred_card.setLayout(batch_pred_layout)
        scroll_layout.addWidget(batch_pred_card)

        scroll_layout.addStretch()
        scroll.setWidget(scroll_widget)
        model_tab_layout.addWidget(scroll)

        self.tabs.addTab(model_tab, "🤖  Load Model")

    def set_status(self, text, level="info"):
        color_map = {
            'success': COLORS['success'],
            'warning': COLORS['warning'],
            'error': COLORS['error'],
            'info': COLORS['text'],
        }
        color = color_map.get(level, COLORS['text'])
        self.status_label.setText(text)
        self.status_label.setStyleSheet(f"color: {color};")

    def update_run_button(self):
        global image_path
        if image_path and os.path.isfile(image_path):
            try:
                with rasterio.open(image_path) as src:
                    if src.count >= 2:
                        self.btn_run.setEnabled(True)
                        return
            except Exception:
                pass
        self.btn_run.setEnabled(False)

    def on_threshold_change(self, value):
        global ndsi_threshold
        ndsi_threshold = value / 100.0
        self.threshold_value_label.setText(f"{ndsi_threshold:.2f}")

    def on_calculation_change(self, text):
        if text == "NDSI":
            self.threshold_label.setText("NDSI Threshold:")
            self.threshold_slider.setMinimum(0)
            self.threshold_slider.setMaximum(100)
            self.threshold_slider.setValue(40)
            self.threshold_slider.setEnabled(True)
        elif text == "NDWI":
            self.threshold_label.setText("NDWI Threshold:")
            self.threshold_slider.setMinimum(-100)
            self.threshold_slider.setMaximum(100)
            self.threshold_slider.setValue(30)
            self.threshold_slider.setEnabled(True)
        elif text == "NDVI":
            self.threshold_label.setText("NDVI Threshold:")
            self.threshold_slider.setMinimum(-100)
            self.threshold_slider.setMaximum(100)
            self.threshold_slider.setValue(20)
            self.threshold_slider.setEnabled(True)
        elif text == "All Indices":
            self.threshold_label.setText("Thresholds (Auto):")
            self.threshold_slider.setEnabled(False)

    # ============================================================
    # EVENT HANDLERS
    # ============================================================

    def upload_folder(self):
        global folder_path, image_files, current_image_index, image_path

        folder = QFileDialog.getExistingDirectory(self, "Select Input Folder")
        if folder:
            folder_path = folder
            extensions = ['.tif', '.tiff', '.TIF', '.TIFF', '.img', '.IMG']
            image_files = [os.path.join(folder_path, f) for f in os.listdir(folder_path)
                           if os.path.splitext(f)[1] in extensions]

            if image_files:
                image_files.sort()
                current_image_index = 0

                self.image_list.clear()
                for img_file in image_files:
                    self.image_list.addItem(os.path.basename(img_file))

                self.folder_label.setText(os.path.basename(folder_path))
                self.folder_label.setStyleSheet(f"color: {COLORS['success']};")
                self.folder_info_label.setText(f"{len(image_files)} GeoTIFF images found")

                image_path = image_files[0]
                self.image_list.setCurrentRow(0)
                self.load_selected_image()
                self.set_status(f"✓ Loaded {len(image_files)} images from folder", "success")
            else:
                self.set_status("✗ No valid images found in folder", "error")
                QMessageBox.warning(self, "No Images", "No valid raster images found in the selected folder.")

    def load_selected_image(self):
        global image_path, current_image_index

        if self.image_list.currentRow() < 0:
            return

        current_image_index = self.image_list.currentRow()
        image_path = image_files[current_image_index]

        try:
            with rasterio.open(image_path) as src:
                band_count = src.count
                width = src.width
                height = src.height
                descriptions = src.descriptions

                band_info_text = f"Bands: {band_count}\n"
                if descriptions:
                    band_info_text += "Band names: " + ", ".join(
                        [desc if desc else f"Band{i + 1}" for i, desc in enumerate(descriptions[:5])])
                    if band_count > 5:
                        band_info_text += "..."

            self.file_name_label.setText(f"Current: {os.path.basename(image_path)}")
            self.file_name_label.setStyleSheet(f"color: {COLORS['success']};")
            self.file_info_label.setText(f"{band_count} bands | {width}x{height} pixels")

            if band_count < 2:
                self.set_status("❌ Error: Image must contain at least 2 bands for index calculation", "error")
                self.btn_run.setEnabled(False)
            else:
                self.set_status(f"✓ Image {current_image_index + 1}/{len(image_files)} loaded – Ready to calculate",
                                "success")
                self.update_run_button()
                self.display_image_preview()

        except Exception as e:
            QMessageBox.critical(self, "Error", f"Failed to load image: {str(e)}")
            self.set_status("❌ Error loading image", "error")

    def clear_image_selection(self):
        global image_path
        self.image_list.clearSelection()
        image_path = None
        self.file_name_label.setText("ALL IMAGES SELECTED (Batch Mode)")
        self.file_name_label.setStyleSheet(f"color: {COLORS['warning']};")
        self.file_info_label.setText("Run 'All Indices' to process all images in folder")
        self.set_status("🔄 Batch mode: Run 'All Indices' to process entire folder", "info")

    def select_output_folder(self):
        global output_folder_path
        folder = QFileDialog.getExistingDirectory(self, "Select Output Folder")
        if folder:
            output_folder_path = folder
            self.output_folder_display.setText(os.path.basename(folder))
            self.output_folder_display.setStyleSheet(f"color: {COLORS['success']};")
            self.set_status(f"✓ Output folder selected: {os.path.basename(folder)}", "success")

    def display_image_preview(self):
        global image_path
        if not image_path:
            return

        try:
            with rasterio.open(image_path) as src:
                data = src.read(1)
                data_min, data_max = np.nanpercentile(data, [2, 98])
                data_norm = np.clip((data - data_min) / (data_max - data_min + 1e-8), 0, 1)
                data_8bit = (data_norm * 255).astype(np.uint8)

                # Convert to QImage
                height, width = data_8bit.shape
                bytes_per_line = width
                q_img = QImage(data_8bit.data, width, height, bytes_per_line, QImage.Format.Format_Grayscale8)
                pixmap = QPixmap.fromImage(q_img)

                # Scale to fit
                scaled_pixmap = pixmap.scaled(
                    self.preview_label.size(),
                    Qt.AspectRatioMode.KeepAspectRatio,
                    Qt.TransformationMode.SmoothTransformation
                )
                self.preview_label.setPixmap(scaled_pixmap)

        except Exception as e:
            self.set_status(f"✗ Preview error: {str(e)}", "error")

    def run_calculation(self):
        calc_type = self.calc_combo.currentText()

        if calc_type == "All Indices":
            if image_path is None:  # Batch mode
                self.batch_process_all_indices()
            else:  # Single image
                self.calculate_all_indices()
        else:
            if image_path is None:  # Batch mode
                self.batch_process_single_index(calc_type)
            else:  # Single image
                self.run_single_calculation(calc_type)

    def run_single_calculation(self, calc_type):
        global image_path, result_data, profile, green, swir, ndsi_threshold

        if not image_path:
            QMessageBox.warning(self, "No Image", "Please load an image first!")
            return

        if not output_folder_path:
            QMessageBox.information(self, "No Output Folder",
                                    "Please select an output folder before running calculation.")
            return

        try:
            self.set_status(f"🔍 Detecting {calc_type} bands...", "warning")
            QApplication.processEvents()

            with rasterio.open(image_path) as src:
                profile = src.profile
                band_result = get_required_bands(src, calc_type)

                if not band_result['success']:
                    self.set_status(f"✗ {band_result['info']}", "error")
                    QMessageBox.critical(self, "Missing Bands", band_result['info'])
                    return

                bands = band_result['bands']

                # Calculate based on type
                if calc_type == 'NDSI':
                    band1_data = bands['GREEN']['data']
                    band2_data = bands['SWIR']['data']
                    result_data = calculate_ndsi(band1_data, band2_data)
                elif calc_type == 'NDWI':
                    band1_data = bands['GREEN']['data']
                    band2_data = bands['NIR']['data']
                    result_data = calculate_ndwi(band1_data, band2_data)
                elif calc_type == 'NDVI':
                    band1_data = bands['NIR']['data']
                    band2_data = bands['RED']['data']
                    result_data = calculate_ndvi(band1_data, band2_data)

            # Calculate statistics
            valid_data = result_data[~np.isnan(result_data)]
            if valid_data.size > 0:
                threshold = ndsi_threshold if calc_type == 'NDSI' else 0.3
                positive_pixels = np.sum(valid_data > threshold)
                total_pixels = valid_data.size
                percentage = (positive_pixels / total_pixels) * 100

                # Save results
                self.save_single_result(calc_type)

                # Update preview
                self.create_preview(result_data, calc_type)

                self.set_status(f"✓ {calc_type} calculated: {percentage:.2f}% above threshold", "success")
            else:
                self.set_status("❌ No valid data", "error")

        except Exception as e:
            self.set_status(f"✗ Error: {str(e)}", "error")
            QMessageBox.critical(self, "Error", f"Calculation failed:\n{str(e)}")

    def calculate_all_indices(self):
        global image_path, profile, output_folder_path

        if not image_path:
            QMessageBox.warning(self, "No Image", "Please load an image first!")
            return

        if not output_folder_path:
            QMessageBox.information(self, "No Output Folder",
                                    "Please select an output folder before running calculation.")
            return

        try:
            self.set_status("🔍 Detecting all required bands...", "warning")
            QApplication.processEvents()

            with rasterio.open(image_path) as src:
                profile = src.profile

                # Detect all bands
                blue_idx = _cached_detect_band(src, 'BLUE')
                green_idx = _cached_detect_band(src, 'GREEN')
                red_idx = _cached_detect_band(src, 'RED')
                nir_idx = _cached_detect_band(src, 'NIR')
                swir_idx = _cached_detect_band(src, 'SWIR')

                missing_bands = []
                if not blue_idx: missing_bands.append('BLUE (B2)')
                if not green_idx: missing_bands.append('GREEN (B3)')
                if not red_idx: missing_bands.append('RED (B4)')
                if not nir_idx: missing_bands.append('NIR (B8)')
                if not swir_idx: missing_bands.append('SWIR (B11)')

                if missing_bands:
                    QMessageBox.critical(self, "Missing Bands",
                                         f"Could not detect the following required bands:\n\n" +
                                         "\n".join([f"• {band}" for band in missing_bands]))
                    self.set_status("❌ Missing required bands", "error")
                    return

                self.set_status("✅ All bands detected! Reading data...", "success")
                QApplication.processEvents()

                # Read bands
                blue = src.read(blue_idx).astype(float)
                green = src.read(green_idx).astype(float)
                red = src.read(red_idx).astype(float)
                nir = src.read(nir_idx).astype(float)
                swir = src.read(swir_idx).astype(float)

            # Calculate indices
            self.set_status("🧮 Calculating NDSI...", "warning")
            QApplication.processEvents()
            with np.errstate(divide='ignore', invalid='ignore'):
                ndsi = (green - swir) / (green + swir)
                ndsi = np.where(np.isfinite(ndsi), ndsi, np.nan)

            self.set_status("🧮 Calculating NDWI...", "warning")
            QApplication.processEvents()
            with np.errstate(divide='ignore', invalid='ignore'):
                ndwi = (green - nir) / (green + nir)
                ndwi = np.where(np.isfinite(ndwi), ndwi, np.nan)

            self.set_status("🧮 Calculating NDVI...", "warning")
            QApplication.processEvents()
            with np.errstate(divide='ignore', invalid='ignore'):
                ndvi = (nir - red) / (nir + red)
                ndvi = np.where(np.isfinite(ndvi), ndvi, np.nan)

            # Save composite
            base_name = os.path.splitext(os.path.basename(image_path))[0]
            composite_path = os.path.join(output_folder_path, f"{base_name}_processed_composite.tiff")

            out_profile = profile.copy()
            out_profile.update(dtype=rasterio.float32, count=8, nodata=np.nan)

            with rasterio.open(composite_path, 'w', **out_profile) as dst:
                dst.write(blue.astype(np.float32), 1)
                dst.write(green.astype(np.float32), 2)
                dst.write(red.astype(np.float32), 3)
                dst.write(nir.astype(np.float32), 4)
                dst.write(swir.astype(np.float32), 5)
                dst.write(ndsi.astype(np.float32), 6)
                dst.write(ndwi.astype(np.float32), 7)
                dst.write(ndvi.astype(np.float32), 8)

                dst.set_band_description(1, 'BLUE (B2)')
                dst.set_band_description(2, 'GREEN (B3)')
                dst.set_band_description(3, 'RED (B4)')
                dst.set_band_description(4, 'NIR (B8)')
                dst.set_band_description(5, 'SWIR (B11)')
                dst.set_band_description(6, 'NDSI (Snow Index)')
                dst.set_band_description(7, 'NDWI (Water Index)')
                dst.set_band_description(8, 'NDVI (Vegetation Index)')

            self.set_status(f"✅ All indices calculated and saved!", "success")
            QMessageBox.information(self, "Success",
                                    f"8-band composite created:\n{composite_path}")

        except Exception as e:
            self.set_status(f"✗ Error: {str(e)}", "error")
            QMessageBox.critical(self, "Error", f"Failed to calculate indices:\n{str(e)}")

    def batch_process_single_index(self, calc_type):
        global folder_path, output_folder_path, image_files

        if not folder_path or not image_files:
            QMessageBox.warning(self, "No Images", "No folder or images loaded!")
            return

        if not output_folder_path:
            QMessageBox.information(self, "No Output Folder",
                                    "Please select an output folder before running calculation.")
            return

        try:
            self.set_status(f"🚀 Batch processing {len(image_files)} images with {calc_type}...", "warning")
            
            # Show progress bar
            self.progress_bar.setMaximum(len(image_files))
            self.progress_bar.setValue(0)
            self.progress_bar.show()
            QApplication.processEvents()

            processed_count = 0
            skipped_count = 0

            for idx, img_file in enumerate(image_files):
                base_name = os.path.splitext(os.path.basename(img_file))[0]
                self.set_status(f"⏳ Processing {calc_type} for image {idx + 1}/{len(image_files)}: {base_name}",
                                "warning")
                
                # Update progress
                self.progress_bar.setValue(idx + 1)
                QApplication.processEvents()

                try:
                    with rasterio.open(img_file) as src:
                        band_result = get_required_bands(src, calc_type)

                        if not band_result['success']:
                            skipped_count += 1
                            continue

                        bands = band_result['bands']

                        if calc_type == 'NDSI':
                            band1_data = bands['GREEN']['data']
                            band2_data = bands['SWIR']['data']
                        elif calc_type == 'NDWI':
                            band1_data = bands['GREEN']['data']
                            band2_data = bands['NIR']['data']
                        elif calc_type == 'NDVI':
                            band1_data = bands['NIR']['data']
                            band2_data = bands['RED']['data']

                        denom = band1_data + band2_data
                        with np.errstate(divide='ignore', invalid='ignore'):
                            result = (band1_data - band2_data) / denom

                        result[denom == 0] = np.nan
                        result = np.clip(result, -1, 1)

                        temp_profile = src.profile.copy()

                    output_filename = f"{base_name}_{calc_type}_processed.tiff"
                    output_path = os.path.join(output_folder_path, output_filename)

                    temp_profile.update(dtype=rasterio.float32, count=1, nodata=np.nan)

                    with rasterio.open(output_path, 'w', **temp_profile) as dst:
                        dst.write(result.astype(np.float32), 1)
                        dst.set_band_description(1, f'{calc_type}')

                    processed_count += 1

                except Exception as e:
                    print(f"Error processing {base_name}: {e}")
                    skipped_count += 1
                    continue

            self.progress_bar.hide()
            self.set_status(f"✅ Batch processing complete! Processed {processed_count} images", "success")
            QMessageBox.information(self, "Batch Complete",
                                    f"Successfully processed: {processed_count} images with {calc_type}\n"
                                    f"Skipped: {skipped_count} images")

        except Exception as e:
            self.progress_bar.hide()
            self.set_status(f"✗ Batch processing failed: {str(e)}", "error")
            QMessageBox.critical(self, "Error", f"Batch processing failed:\n{str(e)}")

    def batch_process_all_indices(self):
        global folder_path, output_folder_path, image_files

        if not folder_path or not image_files:
            QMessageBox.warning(self, "No Images", "No folder or images loaded!")
            return

        if not output_folder_path:
            QMessageBox.information(self, "No Output Folder",
                                    "Please select an output folder before running calculation.")
            return

        try:
            self.set_status(f"🚀 Batch processing {len(image_files)} images with All Indices...", "warning")
            
            # Show progress bar
            self.progress_bar.setMaximum(len(image_files))
            self.progress_bar.setValue(0)
            self.progress_bar.show()
            QApplication.processEvents()

            processed_count = 0
            skipped_count = 0

            for idx, img_file in enumerate(image_files):
                base_name = os.path.splitext(os.path.basename(img_file))[0]
                self.set_status(f"⏳ Processing All Indices for image {idx + 1}/{len(image_files)}: {base_name}",
                                "warning")
                
                # Update progress
                self.progress_bar.setValue(idx + 1)
                QApplication.processEvents()

                try:
                    with rasterio.open(img_file) as src:
                        # Detect all bands
                        blue_idx = _cached_detect_band(src, 'BLUE')
                        green_idx = _cached_detect_band(src, 'GREEN')
                        red_idx = _cached_detect_band(src, 'RED')
                        nir_idx = _cached_detect_band(src, 'NIR')
                        swir_idx = _cached_detect_band(src, 'SWIR')

                        if not all([blue_idx, green_idx, red_idx, nir_idx, swir_idx]):
                            skipped_count += 1
                            continue

                        # Read bands
                        blue = src.read(blue_idx).astype(float)
                        green = src.read(green_idx).astype(float)
                        red = src.read(red_idx).astype(float)
                        nir = src.read(nir_idx).astype(float)
                        swir = src.read(swir_idx).astype(float)

                        # Calculate indices
                        with np.errstate(divide='ignore', invalid='ignore'):
                            ndsi = (green - swir) / (green + swir)
                            ndsi = np.where(np.isfinite(ndsi), ndsi, np.nan)
                            ndwi = (green - nir) / (green + nir)
                            ndwi = np.where(np.isfinite(ndwi), ndwi, np.nan)
                            ndvi = (nir - red) / (nir + red)
                            ndvi = np.where(np.isfinite(ndvi), ndvi, np.nan)

                        composite_profile = src.profile.copy()

                    # Save composite
                    composite_filename = f"{base_name}_processed_composite.tiff"
                    composite_path = os.path.join(output_folder_path, composite_filename)

                    composite_profile.update(dtype=rasterio.float32, count=8, nodata=np.nan)

                    with rasterio.open(composite_path, 'w', **composite_profile) as dst:
                        dst.write(blue.astype(np.float32), 1)
                        dst.write(green.astype(np.float32), 2)
                        dst.write(red.astype(np.float32), 3)
                        dst.write(nir.astype(np.float32), 4)
                        dst.write(swir.astype(np.float32), 5)
                        dst.write(ndsi.astype(np.float32), 6)
                        dst.write(ndwi.astype(np.float32), 7)
                        dst.write(ndvi.astype(np.float32), 8)

                        dst.set_band_description(1, 'BLUE (B2)')
                        dst.set_band_description(2, 'GREEN (B3)')
                        dst.set_band_description(3, 'RED (B4)')
                        dst.set_band_description(4, 'NIR (B8)')
                        dst.set_band_description(5, 'SWIR (B11)')
                        dst.set_band_description(6, 'NDSI (Snow Index)')
                        dst.set_band_description(7, 'NDWI (Water Index)')
                        dst.set_band_description(8, 'NDVI (Vegetation Index)')

                    processed_count += 1

                except Exception as e:
                    print(f"Error processing {base_name}: {e}")
                    skipped_count += 1
                    continue

            self.progress_bar.hide()
            self.set_status(f"✅ Batch processing complete! Processed {processed_count} images", "success")
            QMessageBox.information(self, "Batch Complete",
                                    f"Successfully processed: {processed_count} images\n"
                                    f"Skipped: {skipped_count} images\n\n"
                                    f"Generated {processed_count} composite files (8-band each)")

        except Exception as e:
            self.progress_bar.hide()
            self.set_status(f"✗ Batch processing failed: {str(e)}", "error")
            QMessageBox.critical(self, "Error", f"Batch processing failed:\n{str(e)}")

    def save_single_result(self, calc_type):
        global result_data, profile, image_path, output_folder_path

        if result_data is None or not output_folder_path:
            return

        try:
            base_name = os.path.splitext(os.path.basename(image_path))[0]
            out_tif = os.path.join(output_folder_path, f"{base_name}_{calc_type}_processed.tiff")

            temp_profile = profile.copy()
            temp_profile.update(dtype=rasterio.float32, count=1, nodata=np.nan)

            with rasterio.open(out_tif, 'w', **temp_profile) as dst:
                dst.write(result_data.astype(np.float32), 1)

            self.set_status(f"✓ Saved: {base_name}_{calc_type}_processed.tiff", "success")

        except Exception as e:
            self.set_status(f"✗ Save failed: {str(e)}", "error")

    def create_preview(self, data, calc_type):
        try:
            fig, ax = plt.subplots(figsize=(8, 6), facecolor=COLORS['background'])
            ax.set_facecolor(COLORS['canvas_bg'])

            im = ax.imshow(data, cmap='RdYlGn', vmin=-1, vmax=1)
            ax.set_title(f'{calc_type} Result', color=COLORS['text'], fontsize=14, fontweight='bold')
            ax.axis('off')

            cbar = plt.colorbar(im, ax=ax, orientation='horizontal', pad=0.05, fraction=0.046)
            cbar.set_label('Index Value', color=COLORS['text'])
            cbar.ax.tick_params(colors=COLORS['text'])

            # Save to buffer
            buf = io.BytesIO()
            plt.savefig(buf, format='png', dpi=100, bbox_inches='tight',
                        facecolor=COLORS['background'], edgecolor='none')
            buf.seek(0)
            plt.close(fig)

            # Load and display
            img = PILImage.open(buf)
            img_array = np.array(img)

            height, width, channel = img_array.shape
            bytes_per_line = 3 * width
            q_img = QImage(img_array.data, width, height, bytes_per_line, QImage.Format.Format_RGB888)
            pixmap = QPixmap.fromImage(q_img)

            scaled_pixmap = pixmap.scaled(
                self.preview_label.size(),
                Qt.AspectRatioMode.KeepAspectRatio,
                Qt.TransformationMode.SmoothTransformation
            )
            self.preview_label.setPixmap(scaled_pixmap)

        except Exception as e:
            self.set_status(f"✗ Preview error: {str(e)}", "error")

    # ============================================================
    # MODEL TAB FUNCTIONS
    # ============================================================

    def upload_model(self):
        """Upload a Keras model"""
        global loaded_model, model_path

        file_path, _ = QFileDialog.getOpenFileName(
            self, "Select Keras Model", "",
            "Keras Models (*.keras *.h5);;All Files (*.*)")

        if file_path:
            try:
                import tensorflow as tf
                
                # Custom objects for segmentation models
                custom_objects = {
                    'dice_coef': dice_coef,
                    'dice_loss': dice_loss,
                    'iou_coef': iou_coef,
                }
                
                # Add instance methods as well
                try:
                    custom_objects['dice_coefficient'] = self.dice_coefficient
                    custom_objects['iou_score'] = self.iou_score
                except:
                    pass

                loaded_model = tf.keras.models.load_model(file_path, custom_objects=custom_objects)
                model_path = file_path

                # Update UI
                self.model_filename_display.setText(f"✓ {os.path.basename(file_path)}")
                self.model_filename_display.setStyleSheet(f"""
                    background-color: {COLORS['listbox_bg']};
                    color: {COLORS['success']};
                    padding: 12px;
                    border: 1px solid {COLORS['card_border']};
                    border-radius: 8px;
                """)

                # Get model summary
                import io as io_module
                stream = io_module.StringIO()
                loaded_model.summary(print_fn=lambda x: stream.write(x + '\n'))
                summary = stream.getvalue()

                self.model_details_text.setPlainText(summary)

                self.set_status(f"✓ Model loaded: {os.path.basename(file_path)}", "success")

            except ImportError:
                QMessageBox.critical(self, "TensorFlow Not Found",
                                     "TensorFlow is not installed!\n\nPlease install it using:\npip install tensorflow")
                self.set_status("✗ TensorFlow not found", "error")
            except Exception as e:
                self.set_status(f"✗ Failed to load model: {str(e)}", "error")
                QMessageBox.critical(self, "Error", f"Failed to load model:\n{str(e)}")

    def dice_loss(self, y_true, y_pred, smooth=1e-6):
        """Dice loss function for segmentation"""
        import tensorflow as tf
        y_true_f = tf.keras.backend.flatten(y_true)
        y_pred_f = tf.keras.backend.flatten(y_pred)
        intersection = tf.keras.backend.sum(y_true_f * y_pred_f)
        return 1 - (2. * intersection + smooth) / (tf.keras.backend.sum(y_true_f) + tf.keras.backend.sum(y_pred_f) + smooth)

    def dice_coefficient(self, y_true, y_pred, smooth=1e-6):
        """Dice coefficient metric"""
        import tensorflow as tf
        y_true_f = tf.keras.backend.flatten(y_true)
        y_pred_f = tf.keras.backend.flatten(y_pred)
        intersection = tf.keras.backend.sum(y_true_f * y_pred_f)
        return (2. * intersection + smooth) / (tf.keras.backend.sum(y_true_f) + tf.keras.backend.sum(y_pred_f) + smooth)

    def iou_score(self, y_true, y_pred, smooth=1e-6):
        """IoU metric"""
        import tensorflow as tf
        y_true_f = tf.keras.backend.flatten(y_true)
        y_pred_f = tf.keras.backend.flatten(y_pred)
        intersection = tf.keras.backend.sum(y_true_f * y_pred_f)
        union = tf.keras.backend.sum(y_true_f) + tf.keras.backend.sum(y_pred_f) - intersection
        return (intersection + smooth) / (union + smooth)

    def select_model_input_image(self):
        """Select image for model prediction"""
        global model_input_image_path

        file_path, _ = QFileDialog.getOpenFileName(
            self, "Select Image", "",
            "Images (*.png *.jpg *.jpeg *.tif *.tiff *.bmp);;All Files (*.*)")

        if file_path:
            model_input_image_path = file_path

            # Display preview
            try:
                pixmap = QPixmap(file_path)
                if not pixmap.isNull():
                    scaled_pixmap = pixmap.scaled(
                        self.model_preview_label.size(),
                        Qt.AspectRatioMode.KeepAspectRatio,
                        Qt.TransformationMode.SmoothTransformation
                    )
                    self.model_preview_label.setPixmap(scaled_pixmap)
                    self.set_status(f"✓ Image selected: {os.path.basename(file_path)}", "info")
                else:
                    # Try with rasterio for TIFF files
                    with rasterio.open(file_path) as src:
                        data = src.read(1)
                        data_min, data_max = np.nanpercentile(data, [2, 98])
                        data_norm = np.clip((data - data_min) / (data_max - data_min + 1e-8), 0, 1)
                        data_8bit = (data_norm * 255).astype(np.uint8)

                        height, width = data_8bit.shape
                        q_img = QImage(data_8bit.data, width, height, width, QImage.Format.Format_Grayscale8)
                        pixmap = QPixmap.fromImage(q_img)

                        scaled_pixmap = pixmap.scaled(
                            self.model_preview_label.size(),
                            Qt.AspectRatioMode.KeepAspectRatio,
                            Qt.TransformationMode.SmoothTransformation
                        )
                        self.model_preview_label.setPixmap(scaled_pixmap)
                        self.set_status(f"✓ Image selected: {os.path.basename(file_path)}", "info")

            except Exception as e:
                self.set_status(f"✗ Failed to load image preview: {str(e)}", "error")

    def run_model_prediction(self):
        """Run prediction on selected image"""
        global loaded_model, model_input_image_path, model_prediction_result

        if loaded_model is None:
            QMessageBox.warning(self, "No Model", "Please load a model first.")
            return

        if not model_input_image_path:
            QMessageBox.warning(self, "No Image", "Please select an image first.")
            return

        try:
            import tensorflow as tf

            self.set_status("⏳ Running prediction...", "warning")
            QApplication.processEvents()

            # Get model input shape
            input_shape = loaded_model.input_shape
            target_height = input_shape[1]
            target_width = input_shape[2]
            target_channels = input_shape[3]

            # Load and preprocess image
            img_data = None

            # Try rasterio first for multi-band
            try:
                with rasterio.open(model_input_image_path) as src:
                    if src.count >= target_channels:
                        img_data = np.stack([src.read(i + 1) for i in range(target_channels)], axis=-1)
                    elif src.count > 0:
                        bands = [src.read(i + 1) for i in range(src.count)]
                        while len(bands) < target_channels:
                            bands.append(bands[-1])
                        img_data = np.stack(bands[:target_channels], axis=-1)
            except:
                # Fallback to PIL
                pil_img = PILImage.open(model_input_image_path)
                if pil_img.mode != 'RGB' and target_channels >= 3:
                    pil_img = pil_img.convert('RGB')
                img_data = np.array(pil_img)

            if img_data.ndim == 2:
                img_data = np.expand_dims(img_data, axis=-1)

            # Normalize
            if img_data.dtype != np.uint8:
                img_min, img_max = np.min(img_data), np.max(img_data)
                if img_max > img_min:
                    img_data = ((img_data - img_min) / (img_max - img_min) * 255).astype(np.uint8)
                else:
                    img_data = np.zeros_like(img_data, dtype=np.uint8)

            # Resize
            if img_data.shape[2] <= 3:
                if img_data.shape[2] == 1:
                    pil_img = PILImage.fromarray(img_data[:, :, 0], mode='L')
                elif img_data.shape[2] == 3:
                    pil_img = PILImage.fromarray(img_data, mode='RGB')
                else:
                    pil_img = PILImage.fromarray(img_data[:, :, 0], mode='L')

                pil_img = pil_img.resize((target_width, target_height), PILImage.Resampling.LANCZOS)
                img_array = np.array(pil_img)

                if img_array.ndim == 2:
                    img_array = np.expand_dims(img_array, axis=-1)
            else:
                # Multi-channel resize
                resized_channels = []
                for c in range(min(target_channels, img_data.shape[2])):
                    ch = img_data[:, :, c]
                    pil_ch = PILImage.fromarray(ch, mode='L')
                    pil_ch = pil_ch.resize((target_width, target_height), PILImage.Resampling.LANCZOS)
                    resized_channels.append(np.array(pil_ch))
                img_array = np.stack(resized_channels, axis=2)

            # Handle channel mismatch
            if img_array.shape[2] < target_channels:
                padding = target_channels - img_array.shape[2]
                pad_data = np.repeat(img_array[:, :, -1:], padding, axis=2)
                img_array = np.concatenate([img_array, pad_data], axis=2)
            elif img_array.shape[2] > target_channels:
                img_array = img_array[:, :, :target_channels]

            # Normalize to 0-1
            img_array = img_array.astype(np.float32) / 255.0

            # Add batch dimension
            img_batch = np.expand_dims(img_array, axis=0)

            # Run prediction
            predictions = loaded_model.predict(img_batch, verbose=0)
            model_prediction_result = predictions[0]

            # Handle segmentation output
            if len(model_prediction_result.shape) == 3 and model_prediction_result.shape[2] == 1:
                # Segmentation model
                pred_mask = model_prediction_result[:, :, 0]
                threshold = 0.5
                binary_mask = (pred_mask > threshold).astype(np.uint8)

                total_pixels = binary_mask.size
                positive_pixels = np.sum(binary_mask)
                positive_percentage = (positive_pixels / total_pixels) * 100

                mean_confidence = np.mean(pred_mask)
                max_confidence = np.max(pred_mask)
                min_confidence = np.min(pred_mask)
                std_confidence = np.std(pred_mask)

                result_text = "🎯 SEGMENTATION RESULTS:\n\n"
                result_text += f"📏 Mask Size: {pred_mask.shape[0]} x {pred_mask.shape[1]}\n"
                result_text += f"🎯 Threshold: {threshold}\n\n"
                result_text += "📊 Pixel Statistics:\n"
                result_text += f"  Total Pixels: {total_pixels:,}\n"
                result_text += f"  ✅ Positive: {positive_pixels:,} ({positive_percentage:.2f}%)\n"
                result_text += f"  ❌ Negative: {total_pixels - positive_pixels:,} ({100 - positive_percentage:.2f}%)\n\n"
                result_text += "💯 Confidence Statistics:\n"
                result_text += f"  Mean: {mean_confidence:.4f}\n"
                result_text += f"  Std Dev: {std_confidence:.4f}\n"
                result_text += f"  Max: {max_confidence:.4f}\n"
                result_text += f"  Min: {min_confidence:.4f}\n"

            else:
                # Classification model
                result_text = f"Prediction Shape: {predictions.shape}\n"
                result_text += f"Values:\n{predictions[0]}\n"

                if predictions.shape[1] == 1:
                    prob = predictions[0][0]
                    result_text += f"\nProbability: {prob:.4f}\n"
                    result_text += f"Class: {'Positive' if prob > 0.5 else 'Negative'}"
                elif predictions.shape[1] > 1:
                    class_idx = np.argmax(predictions[0])
                    confidence = predictions[0][class_idx]
                    result_text += f"\nPredicted Class: {class_idx}\n"
                    result_text += f"Confidence: {confidence:.4f}"

            self.model_result_label.setText(result_text)
            self.model_result_label.setStyleSheet(f"""
                background-color: {COLORS['listbox_bg']};
                color: {COLORS['success']};
                padding: 12px;
                border: 1px solid {COLORS['card_border']};
                border-radius: 8px;
            """)

            self.set_status("✓ Prediction complete", "success")

        except Exception as e:
            self.set_status(f"✗ Prediction failed: {str(e)}", "error")
            QMessageBox.critical(self, "Error", f"Prediction failed:\n{str(e)}")
            import traceback
            traceback.print_exc()

    def save_prediction_result(self):
        """Save prediction result to file"""
        global model_prediction_result, model_input_image_path

        if model_prediction_result is None:
            QMessageBox.warning(self, "No Result", "Please run prediction first.")
            return

        try:
            file_path, _ = QFileDialog.getSaveFileName(
                self, "Save Prediction", "",
                "Text Files (*.txt);;NumPy Files (*.npy);;All Files (*.*)")

            if file_path:
                if file_path.endswith('.npy'):
                    np.save(file_path, model_prediction_result)
                else:
                    result_text = self.model_result_label.text()
                    with open(file_path, 'w', encoding='utf-8') as f:
                        f.write(result_text)
                        f.write("\n\n=== Raw Prediction Data ===\n")
                        f.write(str(model_prediction_result))

                self.set_status(f"✓ Prediction saved: {os.path.basename(file_path)}", "success")
                QMessageBox.information(self, "Success", f"Prediction saved to:\n{file_path}")

        except Exception as e:
            self.set_status(f"✗ Save failed: {str(e)}", "error")
            QMessageBox.critical(self, "Error", f"Failed to save prediction:\n{str(e)}")

    def select_batch_folder(self):
        """Select folder for batch predictions"""
        global model_batch_folder

        folder = QFileDialog.getExistingDirectory(self, "Select Batch Folder")
        if folder:
            model_batch_folder = folder
            self.set_status(f"✓ Batch folder selected: {os.path.basename(folder)}", "success")

    def select_model_output_folder(self):
        """Select output folder for model predictions"""
        global model_output_folder

        folder = QFileDialog.getExistingDirectory(self, "Select Output Folder")
        if folder:
            model_output_folder = folder
            self.set_status(f"✓ Model output folder selected: {os.path.basename(folder)}", "success")

    def run_batch_prediction(self):
        """Run batch predictions"""
        global loaded_model, model_batch_folder, model_output_folder

        if loaded_model is None:
            QMessageBox.warning(self, "No Model", "Please load a model first.")
            return

        if not model_batch_folder:
            QMessageBox.warning(self, "No Folder", "Please select a batch folder first.")
            return

        if not model_output_folder:
            QMessageBox.warning(self, "No Output", "Please select an output folder first.")
            return

        try:
            import tensorflow as tf

            # Get all images
            extensions = ['.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff']
            image_files_batch = [f for f in os.listdir(model_batch_folder)
                                 if os.path.splitext(f.lower())[1] in extensions]

            if not image_files_batch:
                QMessageBox.warning(self, "No Images", "No valid images found in the selected folder.")
                return

            # Get model input shape
            input_shape = loaded_model.input_shape
            target_height = input_shape[1]
            target_width = input_shape[2]
            target_channels = input_shape[3]

            success_count = 0
            results = []

            for i, img_file in enumerate(image_files_batch):
                self.set_status(f"Processing {i + 1}/{len(image_files_batch)}: {img_file}", "info")
                QApplication.processEvents()

                try:
                    img_path = os.path.join(model_batch_folder, img_file)

                    # Load and preprocess (simplified version)
                    try:
                        pil_img = PILImage.open(img_path)
                        if pil_img.mode != 'RGB' and target_channels >= 3:
                            pil_img = pil_img.convert('RGB')
                        img_array = np.array(pil_img)
                    except:
                        with rasterio.open(img_path) as src:
                            img_array = src.read(1)

                    if img_array.ndim == 2:
                        img_array = np.expand_dims(img_array, axis=-1)

                    # Resize and normalize
                    if img_array.shape[2] == 1:
                        pil_img = PILImage.fromarray(img_array[:, :, 0].astype(np.uint8), mode='L')
                    else:
                        pil_img = PILImage.fromarray(img_array.astype(np.uint8))

                    pil_img = pil_img.resize((target_width, target_height), PILImage.Resampling.LANCZOS)
                    img_array = np.array(pil_img).astype(np.float32) / 255.0

                    if img_array.ndim == 2:
                        img_array = np.expand_dims(img_array, axis=-1)

                    img_batch = np.expand_dims(img_array, axis=0)

                    # Predict
                    prediction = loaded_model.predict(img_batch, verbose=0)

                    results.append({
                        'filename': img_file,
                        'prediction': prediction[0]
                    })

                    success_count += 1

                except Exception as e:
                    print(f"Error processing {img_file}: {str(e)}")

            # Save results
            output_file = os.path.join(model_output_folder, 'batch_predictions.txt')
            with open(output_file, 'w') as f:
                for result in results:
                    f.write(f"{result['filename']}: {result['prediction']}\n")

            self.set_status(f"✓ Batch complete: {success_count}/{len(image_files_batch)} processed", "success")
            QMessageBox.information(self, "Batch Complete",
                                    f"Successfully processed {success_count} out of {len(image_files_batch)} images.\n\n"
                                    f"Results saved to: {output_file}")

        except Exception as e:
            self.set_status(f"✗ Batch prediction failed: {str(e)}", "error")
            QMessageBox.critical(self, "Error", f"Batch prediction failed:\n{str(e)}")


# ============================================================
# MAIN APPLICATION
# ============================================================
def main():
    app = QApplication(sys.argv)
    app.setApplicationName("Raster Index Calculator")

    font = QFont(_FONT_FAMILY, _FONT_BODY_SIZE)
    app.setFont(font)

    window = MainWindow()
    window.show()

    sys.exit(app.exec())


if __name__ == "__main__":
    main()

qt.qpa.fonts: Populating font family aliases took 70 ms. Replace uses of missing font family "SF Pro Display" with one that exists to avoid this cost. 


SystemExit: 0

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3709: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
